In [ ]:
from sklearn.cluster import KMeans as skKMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import euclidean_distances

from rdkit import Chem
from rdkit.Chem import MolFromSmiles, AllChem
from rdkit import  DataStructs
import numpy as np, pandas as pd

from automol.clustering import ClusteringAlgorithm, MurckoScaffoldClustering, ButinaSplitReassigned, HierarchicalButina, KmeansForSmiles
from automol.stacking import compute_tanimoto_distances
from automol.feature_generators import retrieve_default_offline_generators

def autoMoL_clustering(df,properties,X,clustering,standard_smiles_column='smiles',minority_nb=5,n_clusters=30,cutoff=[0.5],include_chirality=False,random_state=42):
    ########################
    #clustering for groups 
    feature_generators=retrieve_default_offline_generators()
    assert clustering in ['Bottleneck','Butina','HierarchicalButina', 'Scaffold'], 'provide one of Bottleneck or Butina or Scaffold'
    if clustering== 'Scaffold':
        clustering_algorithm=MurckoScaffoldClustering(include_chirality=include_chirality)
    elif clustering== 'Butina':
        clustering_algorithm=ButinaSplitReassigned(cutoff = cutoff)
    elif clustering=='HierarchicalButina':
        if cutoff is not None and not isinstance(cutoff, (list)):
            cutoff=[cutoff, np.minimum(cutoff+0.1,np.maximum(cutoff,0.9))]
        clustering_algorithm=HierarchicalButina(cutoff = cutoff)
    elif clustering== 'Bottleneck':
        clustering_algorithm=KmeansForSmiles(n_groups=n_clusters,feature_generators=feature_generators,used_features=['Bottleneck'],random_state=random_state)
        
    clustering_algorithm.cluster(df[standard_smiles_column])
    df['cluster']=clustering_algorithm.get_groups()
    
    notnull=df[properties].notnull().astype(str).agg('-'.join, axis=1)
    df['stratify'] = notnull+df['cluster'].astype(str) 
    for c in df['stratify'].unique():
        if len(df[df.stratify==c]) < minority_nb:
            df.loc[df.stratify==c, 'stratify']= 'minorities' 
    if len(df[df.stratify=='minorities']) == 1:
        if clustering== 'Bottleneck':
            distance_to_centers=euclidean_distances(X[np.flatnonzero(df.stratify=='minorities')[0],:].reshape(1, -1),kmeans_clustering.cluster_centers_)
            sorted_distances=np.argsort(distance_to_centers[0])
            #only one minority cluster containing one compound, take next closest cluster as cluster
            process_cluster=1
            new_name='minorities'
            while len(df[df.stratify==str(new_name)]) == 1 and process_cluster<len(sorted_distances):
                cluster_prefix=notnull.loc[df.stratify==str(new_name)]
                closest_center=sorted_distances[process_cluster]
                df.loc[df.stratify==str(new_name), 'stratify']=cluster_prefix+str(closest_center)
                new_name=cluster_prefix+str(closest_center)
                process_cluster+=1
        else:
            distance_to_compounds=compute_tanimoto_distances(df.iloc[np.flatnonzero(df.stratify=='minorities')[0]][standard_smiles_column],df[standard_smiles_column])
            closest_compound=np.argsort(distance_to_compounds)[1]
            df.loc[df.stratify=='minorities', 'stratify']= df.iloc[closest_compound]['stratify']     
    return df

In [ ]:
from bokeh.io import output_notebook
output_notebook()
from rdkit import Chem
from rdkit.Chem import MolFromSmiles, AllChem
from rdkit import  DataStructs
from rdkit.Chem import Draw
from rdkit.Chem import PandasTools
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.PandasTools import ChangeMoleculeRendering
from IPython.display import SVG
from bokeh.plotting import figure, show, output_notebook, ColumnDataSource
from bokeh.models import HoverTool
from bokeh.transform import factor_cmap
from bokeh.plotting import figure, output_file, save
from bokeh.models import Title
from bokeh.models import Span
from bokeh.models import Legend, LegendItem

def _prepareMol(mol,kekulize):
    mc = Chem.Mol(mol.ToBinary())
    if kekulize:
        try:
            Chem.Kekulize(mc)
        except:
            mc = Chem.Mol(mol.ToBinary())
    if not mc.GetNumConformers():
        rdDepictor.Compute2DCoords(mc)
    return mc

def moltosvg(mol,molSize=(200,100),kekulize=True,drawer=None,**kwargs):
    mc = _prepareMol(mol,kekulize)
    if drawer is None:
        drawer = rdMolDraw2D.MolDraw2DSVG(molSize[0],molSize[1])
    drawer.DrawMolecule(mc,**kwargs)
    drawer.FinishDrawing()
    svg = drawer.GetDrawingText()
    return SVG(svg.replace('svg:',''))

def get_color(colorscale_name, loc):
    """
    retrieve color based on scale and index
    """
    from _plotly_utils.basevalidators import ColorscaleValidator
    # first parameter: Name of the property being validated
    # second parameter: a string, doesn't really matter in our use case
    cv = ColorscaleValidator("colorscale", "")
    # colorscale will be a list of lists: [[loc1, "rgb1"], [loc2, "rgb2"], ...] 
    colorscale = cv.validate_coerce(colorscale_name)
    
    if hasattr(loc, "__iter__"):
        return [get_continuous_color(colorscale, x) for x in loc]
    return get_continuous_color(colorscale, loc)
        

import plotly.colors
from PIL import ImageColor

def get_continuous_color(colorscale, intermed):
    """
    Plotly continuous colorscales assign colors to the range [0, 1]. This function computes the intermediate
    color for any value in that range.

    Plotly doesn't make the colorscales directly accessible in a common format.
    Some are ready to use:
    
        colorscale = plotly.colors.PLOTLY_SCALES["Greens"]

    Others are just swatches that need to be constructed into a colorscale:

        viridis_colors, scale = plotly.colors.convert_colors_to_same_type(plotly.colors.sequential.Viridis)
        colorscale = plotly.colors.make_colorscale(viridis_colors, scale=scale)

    :param colorscale: A plotly continuous colorscale defined with RGB string colors.
    :param intermed: value in the range [0, 1]
    :return: color in rgb string format
    :rtype: str
    """
    if len(colorscale) < 1:
        raise ValueError("colorscale must have at least one color")

    hex_to_rgb = lambda c: "rgb" + str(ImageColor.getcolor(c, "RGB"))

    if intermed <= 0 or len(colorscale) == 1:
        c = colorscale[0][1]
        return c if c[0] != "#" else hex_to_rgb(c)
    if intermed >= 1:
        c = colorscale[-1][1]
        return c if c[0] != "#" else hex_to_rgb(c)

    for cutoff, color in colorscale:
        if intermed > cutoff:
            low_cutoff, low_color = cutoff, color
        else:
            high_cutoff, high_color = cutoff, color
            break

    if (low_color[0] == "#") or (high_color[0] == "#"):
        # some color scale names (such as cividis) returns:
        # [[loc1, "hex1"], [loc2, "hex2"], ...]
        low_color = hex_to_rgb(low_color)
        high_color = hex_to_rgb(high_color)

    return plotly.colors.find_intermediate_color(
        lowcolor=low_color,
        highcolor=high_color,
        intermed=((intermed - low_cutoff) / (high_cutoff - low_cutoff)),
        colortype="rgb",
    )

def bokeh_cluster_plot(df,title='',add_legend=True,fig_size=(600,600),legend_pos="bottom_right"):
    """
    df['standardized smiles'] -> list of smiles
    df['x'] -> x-coordinates
    df['y'] -> y-coordinates
    df['cluster'] -> labels of clusters
    df['colors'] -> colors
    """
    lw = 1
    
    #smiles_df = pd.DataFrame(df['standardized smiles'].tolist(),columns =['SMILES'])
    PandasTools.AddMoleculeColumnToFrame(df,smilesCol='standardized smiles')
    svgs = np.array([moltosvg(m).data for m in df.ROMol])
    #svgs = np.array([moltosvg(m) for m in df.ROMol])
    ChangeMoleculeRendering(renderer='PNG')


    fig = figure(width=fig_size[0], height=fig_size[1], tools=['reset,box_zoom,wheel_zoom,zoom_in,zoom_out,pan,save']) 
    #fig.add_layout(Title(text=f'{metrics}', text_font_style="italic"), 'above')
    fig.add_layout(Title(text=f'{title}', text_font_style="italic"), 'above')
    fig.add_layout(Title(text=f'Clustering Visualization', text_font_size="16pt"), 'above')
    source = ColumnDataSource(data=dict(x=df['x'], y=df['y'],c=df['colors'],cluster=df['cluster'], desc= df['standardized smiles'],svgs=svgs,index=df.index))
    if add_legend:
        r=fig.scatter('x', 'y', size=7, source=source, name=f'clusters', line_color='black', fill_color='c', fill_alpha=0.5, line_width=0.5,legend_field='cluster')
    else:
        r=fig.scatter('x', 'y', size=7, source=source, name=f'clusters', line_color='black', fill_color='c', fill_alpha=0.5, line_width=0.5)

    hover = HoverTool(tooltips="""
        <div>
            <div> @svgs{safe}
            </div>
            <div>
                <span style="font-size: 8px;">SMILES: @desc</span>
            </div>
            <div>
                <span style="font-size: 10px; font-weight: bold;">Cluster: @cluster</span>
            </div>
            <div>
                <span style="font-size: 10px; ">Dataframe index: @index</span>
            </div>
        </div>
        """
    ,renderers=[r])

    fig.add_tools(hover)
    
    fig.background_fill_color = "lightgray"
    fig.background_fill_alpha = 0.3
    fig.legend.border_line_width = 2
    fig.legend.border_line_color = "black"
    fig.legend.background_fill_color = "lightgray"
    fig.legend.background_fill_alpha = 0.4                     
    fig.legend.label_text_font_size = "8px"
    
    fig.xaxis.axis_label ="X-coordinate"
    fig.yaxis.axis_label ="Y-coordinate"
    fig.legend.location = legend_pos

    return fig

In [ ]:
#####################################################
#name of the dataset
name='Caco2_Wang'
#name='HydrationFreeEnergy_FreeSolv'
#name='Lipophilicity_AstraZeneca'
#name='PPBR_AZ'
#set the column in the data file where the smiles can be found
smiles_column='Drug'
#verbosity
verbose=1
# set to true if you want to add rdkit descriptors to the feature set
check_rdkit_desc=False
#name of the new column to add the standardized smiles in the dataframe (choice is free)
standard_smiles_column='stand_SMILES'

properties=['Y']
#####################################################
from tdc.single_pred import ADME
import numpy as np, pandas as pd
from  matplotlib import pyplot as plt
from automol.property_prep import add_stereo_smiles,validate_rdkit_smiles, add_rdkit_standardized_smiles
#dictionary to keep track of the relevant training parameters

data = ADME(name = name)
split = data.get_split()
df=split['train']
test_df=split['test']
df.head()

model_param={}
model_param['Data file']=name

#in case of reading data from csv
#df= pd.read_csv(file_name, na_values = ['NAN', '?','NaN'])

add_rdkit_standardized_smiles(df, smiles_column,verbose=verbose,outname=standard_smiles_column)
add_rdkit_standardized_smiles(test_df, smiles_column,verbose=verbose,outname=standard_smiles_column)

model_param['Standardization']='rdkit'

if check_rdkit_desc: 
    validate_rdkit_smiles(df, standard_smiles_column,verbose=verbose)

df.dropna(inplace=True, subset = [standard_smiles_column])
df.head(5)

In [ ]:
from automol.feature_generators import retrieve_default_offline_generators
from automol.property_prep import add_stereo_smiles,validate_rdkit_smiles, add_rdkit_standardized_smiles
##################################################
rdkit_standardization=True
# Scaffold, Bottleneck, Butina or HierarchicalButina
clustering='Bottleneck'
# threshold for small clusters
minority_nb=5
# k value for kmeans
n_clusters=20
# cutoff for butina
cutoff=[0.5]
# chirality for MurckoScaffold
include_chirality=False
###################################################

if rdkit_standardization:
    add_rdkit_standardized_smiles(df, smiles_column,verbose=1,outname='standardized smiles')
else:
    add_stereo_smiles(df,smiles_column,verbose=1,outname='standardized smiles')

df.dropna(inplace=True, subset = ['standardized smiles'])
df.dropna(inplace=True, subset = properties,how='all')
df.head()
#dictionary holding the default feature generators
feature_generators=retrieve_default_offline_generators(radius=2,nbits=2048)

smiles_list=df['standardized smiles'].values
X=feature_generators['Bottleneck'].generate(smiles_list)

df=autoMoL_clustering(df,properties,X,clustering=clustering,standard_smiles_column='standardized smiles',minority_nb=minority_nb,n_clusters=n_clusters,cutoff=cutoff,include_chirality=include_chirality)
df.head()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
unique_clusters=df['stratify'].unique()
###################################################
#color map
cmap='jet'
color_dict={c:get_color(cmap, (float(i) / len(unique_clusters))) for i,c in enumerate(unique_clusters)}
#only show legend if nb of clusters is less than 15
add_legend=len(unique_clusters)<=25
###################################################

#reduce to 50 with pca for speed
pca_50 = PCA(n_components=50)
X_50 = pca_50.fit_transform(X)
# Compute TSNE
X_embedded = TSNE(n_components=2, learning_rate='auto',init='random', perplexity=30).fit_transform(X_50)
df['x']=X_embedded[:,0]
df['y']=X_embedded[:,1]
df['cluster'] = df['stratify']
df['colors'] = [ color_dict[c] for c in df['stratify']]
bokeh_fig=bokeh_cluster_plot(df,title=f'Visualizing {clustering}-clustering using PCA(50) before TSNE',fig_size=(1200,800),add_legend=add_legend,legend_pos="bottom_right")
show(bokeh_fig)